In [1]:
import xmlrpc.client
from datetime import datetime

# =========================================================================
# 1. ENVIRONMENT INITIALIZATION & SERVER PROXY CONNECTION
# =========================================================================
URL = "http://localhost:8069"
DB = "m.nushath"       
USERNAME = "user@company.com"  
API_KEY = "user"              

print("Initializing XML-RPC proxies...")
common = xmlrpc.client.ServerProxy(f'{URL}/xmlrpc/2/common', allow_none=True)
models = xmlrpc.client.ServerProxy(f'{URL}/xmlrpc/2/object', allow_none=True)

uid = common.authenticate(DB, USERNAME, API_KEY, {})
if not uid:
    print("[CRITICAL] Authentication failed! Check connection properties.")
    exit()

print(f"[SUCCESS] Connected securely as User ID: {uid}\n" + "="*70)

Initializing XML-RPC proxies...
[SUCCESS] Connected securely as User ID: 5


In [2]:
# =========================================================================
# DYNAMIC PREREQUISITE DATA RESOLVER (Ensures the Demo Runs Error-Free)
# =========================================================================
# Fetch user company access rules dynamically
user_data = models.execute_kw(DB, uid, API_KEY, 'res.users', 'read', [[uid]], {'fields': ['company_id', 'company_ids']})
current_company_id = user_data[0]['company_id'][0]
allowed_company_ids = user_data[0]['company_ids']

# Locate an existing product variant to use for our transactional sub-lines
product_search = models.execute_kw(DB, uid, API_KEY, 'product.product', 'search', [[['sale_ok', '=', True]]], {'limit': 1})
# Locate any existing product tags to demonstrate the Many2many assignment array
tag_search = models.execute_kw(DB, uid, API_KEY, 'product.tag', 'search', [[]], {'limit': 3})

target_product_id = product_search[0] if product_search else False
target_tag_ids = tag_search if tag_search else []

In [4]:
# =========================================================================
# DEMO 1: STANDARD FIELD CREATION (PRIMITIVE TYPES)
# Documentation Link: Recipe #1 - Standalone Contact Cards
# =========================================================================
print("[DEMO 1] Executing Primitive Type Creation...")

primitive_vals = {
    'name': 'Acme Global Ltd (API Demo)',
    'is_company': True,
    'email': 'info@acmeglobal.demo',
    'phone': '+1-555-0199',
    'website': 'https://www.acmeglobal.demo',
    'company_id': current_company_id
}

# Payload is passed as an object inside a single element array: [vals_dict]
parent_company_id = models.execute_kw(DB, uid, API_KEY, 'res.partner', 'create', [primitive_vals])

print(f" -> [SUCCESS] Created Parent Company Contact. Assigned ID: {parent_company_id}\n" + "-"*50)

[DEMO 1] Executing Primitive Type Creation...
 -> [SUCCESS] Created Parent Company Contact. Assigned ID: 14
--------------------------------------------------


In [6]:
# =========================================================================
# DEMO 2: MANY2ONE RELATIONAL CREATION (LINKING TO A PARENT)
# Documentation Link: Recipe #2 - Linking Contacts to Parents
# =========================================================================
print("[DEMO 2] Executing Many2one Link Creation...")

many2one_vals = {
    'name': 'John Doe (API Representative)',
    'is_company': False,
    'parent_id': parent_company_id,  # Foreign Key integer assigned directly
    'email': 'j.doe@acmeglobal.demo',
    'company_id': current_company_id
}

child_contact_id = models.execute_kw(DB, uid, API_KEY, 'res.partner', 'create', [many2one_vals])
print(f" -> [SUCCESS] Linked Child Contact ID {child_contact_id} directly to Parent Company ID {parent_company_id}\n" + "-"*50)

[DEMO 2] Executing Many2one Link Creation...
 -> [SUCCESS] Linked Child Contact ID 16 directly to Parent Company ID 14
--------------------------------------------------


In [8]:
# =========================================================================
# DEMO 3A: ONE2MANY SPECIAL COMMAND CREATION (PARENT + SUB-LINES)
# Documentation Link: Recipe #3.A - Sales Orders with Nested Line Items
# =========================================================================
print("[DEMO 3A] Executing One2many Nested Record Line Items Creation...")

if not target_product_id:
    print(" -> Skipped: No product variants available in database to construct order line sub-objects.")
else:
    one2many_vals = {
        'partner_id': parent_company_id,
        'company_id': current_company_id,
        'date_order': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'note': 'Order and nested lines instantiated in a single database execution block.',
        
        # Using the special triplet format: (0, 0, {values_dictionary})
        'order_line': [
            (0, 0, {
                'product_id': target_product_id,
                'product_uom_qty': 5.0,
                'price_unit': 45.00,
                'name': 'API Demonstration Item Alpha'
            }),
            (0, 0, {
                'product_id': target_product_id,
                'product_uom_qty': 12.0,
                'price_unit': 12.50,
                'name': 'API Demonstration Item Beta'
            })
        ]
    }

    order_id = models.execute_kw(DB, uid, API_KEY, 'sale.order', 'create', [one2many_vals])
    print(f" -> [SUCCESS] Transactional Order generated natively. Created Order ID: {order_id}\n" + "-"*50)

[DEMO 3A] Executing One2many Nested Record Line Items Creation...
 -> [SUCCESS] Transactional Order generated natively. Created Order ID: 59
--------------------------------------------------


In [11]:
# =========================================================================
# DEMO 3B: MANY2MANY SPECIAL COMMAND CREATION (REPLACE ALL LINKED ENTRIES)
# Documentation Link: Recipe #3.B - Product Templates with Assigned Tags
# =========================================================================
print("[DEMO 3B] Executing Many2many Link Replacement Array Command...")

many2many_vals = {
    'name': 'Enterprise Gateway Core Module',
    'list_price': 1250.00,
    'type': 'consu',
    # Command triplet syntax (6, 0, [ID_list]) forces database to link exactly these relations
    'product_tag_ids': [(6, 0, target_tag_ids)] if target_tag_ids else []
}

product_template_id = models.execute_kw(DB, uid, API_KEY, 'product.template', 'create', [many2many_vals])
print(f" -> [SUCCESS] Created Product ID: {product_template_id} linked with Tag IDs: {target_tag_ids}\n" + "-"*50)

[DEMO 3B] Executing Many2many Link Replacement Array Command...
 -> [SUCCESS] Created Product ID: 12 linked with Tag IDs: []
--------------------------------------------------


In [12]:
# =========================================================================
# DEMO 4: BATCH PERFORMANCE CREATION
# Documentation Link: Recipe #4 - Multi-Record Arrays
# =========================================================================
print("[DEMO 4] Executing Batch Mass Array Insertion...")

# Wrap a collection list of dictionary objects into a matrix array parameter: [[dict1, dict2, dict3]]
batch_payload = [
    {'name': 'Mass Sync Location East', 'comment': 'API Automated Bulk Feed Grid A', 'company_id': current_company_id},
    {'name': 'Mass Sync Location West', 'comment': 'API Automated Bulk Feed Grid B', 'company_id': current_company_id},
    {'name': 'Mass Sync Location North', 'comment': 'API Automated Bulk Feed Grid C', 'company_id': current_company_id}
]

batch_created_ids = models.execute_kw(DB, uid, API_KEY, 'res.partner', 'create', [batch_payload])
print(f" -> Method output data type: {type(batch_created_ids)} (Array of integers)")
print(f" -> [SUCCESS] Batch transaction finalized. Generated Database record references: {batch_created_ids}\n" + "-"*50)

[DEMO 4] Executing Batch Mass Array Insertion...
 -> Method output data type: <class 'list'> (Array of integers)
 -> [SUCCESS] Batch transaction finalized. Generated Database record references: [17, 18, 19]
--------------------------------------------------


In [13]:
# =========================================================================
# DEMO 5: EXPLICIT MULTI-COMPANY CONTEXT CREATION
# Documentation Link: Recipe #5 - Keyword Arguments Context Injections
# =========================================================================
print("[DEMO 5] Executing Isolated Multi-Company Context Creation...")

# Structure the secure system parameters layout
explicit_multi_company_context = {
    'company_id': current_company_id,
    'allowed_company_ids': allowed_company_ids
}

context_vals = {
    'name': 'Automated API Cash Registry',
    'code': 'APICR',
    'type': 'cash',
    'company_id': current_company_id
}

try:
    # Notice context is explicitly passed as a key inside the standard options dictionary parameter block
    custom_context_journal_ids = models.execute_kw(DB, uid, API_KEY, 'account.journal', 'create', 
        [context_vals], 
        {'context': explicit_multi_company_context}
    )
    print(f" -> [SUCCESS] Multi-Tenant Guard bypassed. Safe Journal Record Created with ID: {custom_context_journal_ids[0]}")
except Exception as e:
    print(f" -> Context Engine Blocked Creation: {e}. (Verify user accounting admin group authorization permissions)")

print("\n" + "="*70 + "\n[FINISHED] Complete modular code creation demonstration run successfully.")

[DEMO 5] Executing Isolated Multi-Company Context Creation...
 -> Context Engine Blocked Creation: 'int' object is not subscriptable. (Verify user accounting admin group authorization permissions)

[FINISHED] Complete modular code creation demonstration run successfully.
